# Generating the Data Frame

In [25]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_rows = 10000

# Base time array with daylight saving jumps and irregular intervals
base_times = pd.date_range("2024-03-01", periods=n_rows, freq="5min")

# 1. Multi-format and corrupted datetime column
raw_timestamps = []
for i, dt in enumerate(base_times):
    r = i % 10
    if r == 0:
        raw_timestamps.append(dt.strftime("%Y-%m-%d %H:%M:%S"))
    elif r == 1:
        raw_timestamps.append(dt.strftime("%d/%m/%Y %I:%M %p"))
    elif r == 2:
        raw_timestamps.append(dt.strftime("%b %d, %Y %H:%M"))
    elif r == 3:
        raw_timestamps.append(str(int(dt.timestamp())))  # Unix epoch
    elif r == 4:
        raw_timestamps.append(dt.strftime("%Y.%m.%d-%H.%M.%S"))
    elif r == 5:
        raw_timestamps.append("2024-02-30 14:00:00")     # Impossible calendar date
    elif r == 6:
        raw_timestamps.append("ERR_CORRUPT_STAMP")       # Text garbage
    elif r == 7:
        raw_timestamps.append("")                        # Empty string
    else:
        raw_timestamps.append(dt.strftime("%Y-%m-%dT%H:%M:%SZ"))

# 2. Inconsistent Station IDs & Location Metadata
station_pool = [
    "STATION_A", "station_a", " STATION_A ", "STA-A",
    "STATION_B", "station-b", "STATION_B\t",
    "STATION_C", "UNKNOWN", None, np.nan
]
stations = np.random.choice(station_pool, size=n_rows)

# 3. Numeric column with mixed currency/units, commas, and negative flags
power_draw = np.random.normal(loc=450.0, scale=45.0, size=n_rows).round(2).astype(str)
for i in range(n_rows):
    if i % 15 == 0:
        power_draw[i] = f"{float(power_draw[i]):,.2f} kW"
    elif i % 23 == 0:
        power_draw[i] = f"${power_draw[i]}"
    elif i % 47 == 0:
        power_draw[i] = "OFFLINE"
    elif i % 73 == 0:
        power_draw[i] = "-9999"

# 4. Sensor Temperature: Sentinels, physical impossibilities, and drift
temp = np.random.normal(loc=24.0, scale=3.5, size=n_rows)
temp[::150] = -999.0     # Missing code sentinel
temp[::300] = 1250.0     # Sensor voltage spike
temp[::450] = -273.15    # Absolute zero glitch
temp[2000:2080] = np.nan # Multi-hour signal blackout

# 5. Pressure with stuck sensors (zero variance runs)
pressure = np.random.normal(loc=1013.25, scale=5.0, size=n_rows)
pressure[4000:4300] = 1013.25  # Dead sensor flatlining for 25 hours

# 6. JSON-encoded metadata payload (nested semi-structured data)
payloads = []
for i in range(n_rows):
    if i % 10 == 0:
        payloads.append('{"firmware": "v1.2", "battery_pct": "88%"}')
    elif i % 25 == 0:
        payloads.append('{"firmware": "v2.0-beta", "battery_pct": "LOW"}')
    elif i % 50 == 0:
        payloads.append("CORRUPTED_JSON_BLOB")
    else:
        payloads.append(f'{{"firmware": "v1.0", "battery_pct": "{np.random.randint(50, 100)}%"}}')

# Assemble DataFrame
df_messy = pd.DataFrame({
    "recorded_at": raw_timestamps,
    "station_code": stations,
    "power_kwh": power_draw,
    "temp_celsius": temp,
    "baro_pressure_hpa": pressure,
    "diagnostic_payload": payloads
})

# 7. Add duplicate rows with minor timestamp jitter
duplicates = df_messy.iloc[500:600].copy()
df_messy = pd.concat([df_messy, duplicates]).sample(frac=1, random_state=42).reset_index(drop=True)

df_messy.to_csv("iot_telemetry_dirty.csv", index=False)
print(f"Created 'iot_telemetry_dirty.csv' with shape {df_messy.shape}")

Created 'iot_telemetry_dirty.csv' with shape (10100, 6)


In [26]:
df_messy[:25]

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,26/03/2024 04:45 PM,STA-A,449.84,20.539694,1012.775900,"{""firmware"": ""v1.0"", ""battery_pct"": ""50%""}"
1,2024.03.21-06.10.00,STATION_B\t,477.23,26.659833,1008.006831,"{""firmware"": ""v1.0"", ""battery_pct"": ""53%""}"
2,"Mar 08, 2024 08:50",station-b,470.45,25.275140,1013.868000,"{""firmware"": ""v1.0"", ""battery_pct"": ""75%""}"
3,2024-03-31T12:20:00Z,station_a,428.42,19.337494,1020.006567,"{""firmware"": ""v1.0"", ""battery_pct"": ""63%""}"
4,2024.03.02-01.20.00,STATION_B\t,445.72,25.045033,1005.984500,"{""firmware"": ""v1.0"", ""battery_pct"": ""68%""}"
5,09/03/2024 06:55 PM,STA-A,472.04,22.961166,1005.664661,"{""firmware"": ""v1.0"", ""battery_pct"": ""61%""}"
6,2024-02-30 14:00:00,UNKNOWN,384.29,28.383664,1014.747788,"{""firmware"": ""v1.0"", ""battery_pct"": ""62%""}"
7,2024-03-27T14:15:00Z,station-b,$427.06,26.718220,1010.547397,"{""firmware"": ""v1.0"", ""battery_pct"": ""97%""}"
8,2024.03.29-13.20.00,STATION_A,428.56,27.578694,1019.435783,"{""firmware"": ""v1.0"", ""battery_pct"": ""82%""}"
9,2024-03-16T10:40:00Z,STA-A,438.64,20.279884,1019.263711,"{""firmware"": ""v1.0"", ""battery_pct"": ""72%""}"


# Step 1: Dropping duplicates and fixing `recorded_at` column

It's a good idea to drop dulicate rows first to avoid unnecessary work later on. We'll then work on the `recorded_at` column.

There are several problems with the data in the `recorded_at` column. The most obvious one is that the dates are formatted differently to each other, making it difficult to manipulate the data. Another obvious one is that some values are missing or have been corrupted (`ERR_CORRUPT_STAMP`); it would be convenient to replace them with `pd.NaT`.

There are several more problems.
* Given that all the dates seem to be in March, it seems clear that the values which use `*/*/*` for the date (e.g. `02/03/2024`) are using the UK convention of the day first. This must be taken into account when converting the dates to UTC format.
* Some dates are given as Unix time in 10 digits.
* Some values use `*.*.*-*.*.*` for the date and time, which is non-standard and will make it difficult for `pd.to_datetime` to apply to it.
* Some values are 30th February 2024, which is an impossible date, so they should be replaced by `pd.NaT` too.

Our goal will be to convert all the different times to `datetime` objects and replace corrupt or faulty time stamps with `pd.NaT`.

In [42]:
df1 = df_messy.copy()

df1 = df1.drop_duplicates()

def clean_timestamps(series: pd.Series) -> pd.Series:
    # 1. Clean whitespace and normalize custom delimiter *.*.*-*.*.*
    normalised = (
        series.astype(str)
        .str.strip()
        .str.replace(r"^(\d{4})\.(\d{2})\.(\d{2})-(\d{2})\.(\d{2})\.(\d{2})$", r"\1-\2-\3 \4:\5:\6", regex = True)
    )

    # 2. Extract Unix epoch strings convert to epoch seconds

    unix_bool = normalised.str.match(r"^\d{10}$")
    cleaned_unix = pd.to_datetime(
        pd.to_numeric(normalised.where(unix_bool), errors = "coerce"),
    unit = "s",
    errors = "coerce",
    utc = True
    )

    # 3. Parse remaining strings with dayfirst=True to protect DD/MM/YYYY dates. This stage also deals with all empty or corrupt values.

    cleaned_dates = pd.to_datetime(
        normalised.where(~unix_bool),
        format = "mixed",
        dayfirst = True,
        errors = "coerce",
        utc = True
    )

    # Combine: fill unix timestamps into the main series
    return cleaned_dates.combine_first(cleaned_unix)

# Apply to DataFrame
df1["recorded_at"] = clean_timestamps(df1["recorded_at"])

Then we sort the dataframe by time and reset the index.

In [29]:
df1 = df1.sort_values(by = 'recorded_at').reset_index(drop = True)
df1

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,2024-03-01 00:00:00+00:00,STATION_B\t,413.61 kW,-273.150000,1016.023518,"{""firmware"": ""v1.2"", ""battery_pct"": ""88%""}"
1,2024-03-01 00:05:00+00:00,STA-A,535.03,24.777070,1017.724282,"{""firmware"": ""v1.0"", ""battery_pct"": ""56%""}"
2,2024-03-01 00:10:00+00:00,NaN,431.52,28.232964,1011.302091,"{""firmware"": ""v1.0"", ""battery_pct"": ""64%""}"
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,"{""firmware"": ""v1.0"", ""battery_pct"": ""70%""}"
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,"{""firmware"": ""v1.0"", ""battery_pct"": ""74%""}"
...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,"{""firmware"": ""v1.0"", ""battery_pct"": ""60%""}"
9996,NaT,STA-A,480.22,24.455721,1009.338727,"{""firmware"": ""v1.0"", ""battery_pct"": ""76%""}"
9997,NaT,STATION_A,470.45,24.299476,1013.216474,"{""firmware"": ""v1.0"", ""battery_pct"": ""91%""}"
9998,NaT,STATION_C,363.91,22.642465,1014.534921,"{""firmware"": ""v1.0"", ""battery_pct"": ""54%""}"


# Step 2: Fixing `station_code` column

The data is taken from three stations: Station A, Station B and Station C. However, the values of the `station_code` column are formatted differently to each other. They take the values `STATION_A`, `station_a`, ` STATION_A ` (with a space before and after), `STA-A`, `STATION_B`, `station-b`, `STATION_B\t` and `STATION_C`, as well as `UNKNOWN`, `None` and `NaN`.

We will replace all instances of Stations A and B with `STATION_A` and `STATION_B` respectively (all instances of Station C are already set to `STATION_C`), and all instances of `None` and `NaN` to `UNKNOWN`.

In [32]:
df2 = df1.copy()

mapping = {'STA_A': 'STATION_A'}

df2['station_code'] = df2['station_code'].fillna('UNKNOWN').astype(str).str.strip().str.upper().str.replace("-", "_", regex = False).replace(mapping)

df2

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,2024-03-01 00:00:00+00:00,STATION_B,413.61 kW,-273.150000,1016.023518,"{""firmware"": ""v1.2"", ""battery_pct"": ""88%""}"
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,"{""firmware"": ""v1.0"", ""battery_pct"": ""56%""}"
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,"{""firmware"": ""v1.0"", ""battery_pct"": ""64%""}"
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,"{""firmware"": ""v1.0"", ""battery_pct"": ""70%""}"
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,"{""firmware"": ""v1.0"", ""battery_pct"": ""74%""}"
...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,"{""firmware"": ""v1.0"", ""battery_pct"": ""60%""}"
9996,NaT,STATION_A,480.22,24.455721,1009.338727,"{""firmware"": ""v1.0"", ""battery_pct"": ""76%""}"
9997,NaT,STATION_A,470.45,24.299476,1013.216474,"{""firmware"": ""v1.0"", ""battery_pct"": ""91%""}"
9998,NaT,STATION_C,363.91,22.642465,1014.534921,"{""firmware"": ""v1.0"", ""battery_pct"": ""54%""}"


# Step 3: Fixing the `power_kwh` column

There are 2 problems with the data in . Some values have `kW` or `$` as units, even though the correct unit is `kWh`, as specified by the column name. These units need to be removed. Additionally, some values are `-9999`, which is clearly an error, and it is inconvenient to have a string `'OFFLINE'` in a column containing floats, so all `-9999` and `'OFFLINE'` values will be converted to `NaN`s.

We will make use of the fact that whenever `$` appears, it is the first character of the value with no space between it and the number, and whenver `kW` appears, it is the final two characters of the value with a space before it.

In [41]:
df3 = df2.copy()
power_clean = df3['power_kwh'].astype(str).str.replace(r"[$,]|kW|\s", "", regex = True)
df3['power_kwh'] = pd.to_numeric(power_clean, errors= 'coerce').replace(-9999, np.nan)
df3

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,2024-03-01 00:00:00+00:00,STATION_B,413.61,-273.150000,1016.023518,"{""firmware"": ""v1.2"", ""battery_pct"": ""88%""}"
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,"{""firmware"": ""v1.0"", ""battery_pct"": ""56%""}"
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,"{""firmware"": ""v1.0"", ""battery_pct"": ""64%""}"
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,"{""firmware"": ""v1.0"", ""battery_pct"": ""70%""}"
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,"{""firmware"": ""v1.0"", ""battery_pct"": ""74%""}"
...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,"{""firmware"": ""v1.0"", ""battery_pct"": ""60%""}"
9996,NaT,STATION_A,480.22,24.455721,1009.338727,"{""firmware"": ""v1.0"", ""battery_pct"": ""76%""}"
9997,NaT,STATION_A,470.45,24.299476,1013.216474,"{""firmware"": ""v1.0"", ""battery_pct"": ""91%""}"
9998,NaT,STATION_C,363.91,22.642465,1014.534921,"{""firmware"": ""v1.0"", ""battery_pct"": ""54%""}"


# Step 4: Fixing the `temp_celsius` column

Most of the `temp_celsius` column is fine. However, sometimes its values are one of `-999.0`, `1250.0` and `-273.15`, which can't be right: `-999.0` and `-273.15` aren't physically possible, and `1250.0` is clearly far too high when the other values are between `20` and `30`.

In [44]:
df4 = df3.copy()

# Set a reasonable limit on physically plausible temperatures
temp_interval = (-50, 60)
df4['temp_celsius'] = df4['temp_celsius'].where(df4['temp_celsius'].between(*temp_interval))
df4

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,2024-03-01 00:00:00+00:00,STATION_B,413.61,NaN,1016.023518,"{""firmware"": ""v1.2"", ""battery_pct"": ""88%""}"
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,"{""firmware"": ""v1.0"", ""battery_pct"": ""56%""}"
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,"{""firmware"": ""v1.0"", ""battery_pct"": ""64%""}"
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,"{""firmware"": ""v1.0"", ""battery_pct"": ""70%""}"
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,"{""firmware"": ""v1.0"", ""battery_pct"": ""74%""}"
...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,"{""firmware"": ""v1.0"", ""battery_pct"": ""60%""}"
9996,NaT,STATION_A,480.22,24.455721,1009.338727,"{""firmware"": ""v1.0"", ""battery_pct"": ""76%""}"
9997,NaT,STATION_A,470.45,24.299476,1013.216474,"{""firmware"": ""v1.0"", ""battery_pct"": ""91%""}"
9998,NaT,STATION_C,363.91,22.642465,1014.534921,"{""firmware"": ""v1.0"", ""battery_pct"": ""54%""}"


# Step 5: Fixing the `baro_pressure_hpa` column

The only problem with the `baro_pressure_hpa` column is that 300 consecutive rows had value `1013.25`, which is clearly a bug. We will therefore replace the faulty values with `np.nan`.

In [45]:
df5 = df4.copy()
df5[df5['baro_pressure_hpa'] == 1013.25].shape

(300, 6)

Conveniently, precisely 300 rows in our current data frame have value `1013.25` in the `baro_pressure_hpa` column, so precisely those values should be replaced with `NaN`.

In [49]:
df5['baro_pressure_hpa'] = df5['baro_pressure_hpa'].replace(1013.25, np.nan)
df5

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,diagnostic_payload
0,2024-03-01 00:00:00+00:00,STATION_B,413.61,NaN,1016.023518,"{""firmware"": ""v1.2"", ""battery_pct"": ""88%""}"
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,"{""firmware"": ""v1.0"", ""battery_pct"": ""56%""}"
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,"{""firmware"": ""v1.0"", ""battery_pct"": ""64%""}"
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,"{""firmware"": ""v1.0"", ""battery_pct"": ""70%""}"
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,"{""firmware"": ""v1.0"", ""battery_pct"": ""74%""}"
...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,"{""firmware"": ""v1.0"", ""battery_pct"": ""60%""}"
9996,NaT,STATION_A,480.22,24.455721,1009.338727,"{""firmware"": ""v1.0"", ""battery_pct"": ""76%""}"
9997,NaT,STATION_A,470.45,24.299476,1013.216474,"{""firmware"": ""v1.0"", ""battery_pct"": ""91%""}"
9998,NaT,STATION_C,363.91,22.642465,1014.534921,"{""firmware"": ""v1.0"", ""battery_pct"": ""54%""}"


# Step 6: Fixing the `diagnostic_payload` column

There are several problems with the `diagnostic_payload` column:
1. Whenever the `firmware` is `v2.0-beta`, the `battery_pct` value is given as `LOW`, while it should be an integer percentage.
2. Sometimes a `CORRUPTED_JSON_BLOB` is given.
3. In general, each value is JSON-encoded, not expressed as a dictionary.
4. It would be convenient to split the column into two columns: `firmware` and `battery_pct`.

First we will change all `CORRUPTED_JSON_BLOB` entries to `NaN`s. We will then convert each non-NaN value to a dictionary, split the column into 2 columns, and replace `LOW` with `NaN`.

In [52]:
import json 

df6 = df5.copy()

df6['diagnostic_payload'] = df6['diagnostic_payload'].replace('CORRUPTED_JSON_BLOB', np.nan)

diagnostic_json_to_dict = df6['diagnostic_payload'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})

new_diagnostic_cols = pd.json_normalize(diagnostic_json_to_dict)

df6 = df6.drop(columns = 'diagnostic_payload')

df6 = pd.concat([df6, new_diagnostic_cols], axis = 1)
df6['battery_pct'] = df6['battery_pct'].replace('LOW', np.nan)

df6

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,firmware,battery_pct
0,2024-03-01 00:00:00+00:00,STATION_B,413.61,NaN,1016.023518,v1.2,88%
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,v1.0,56%
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,v1.0,64%
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,v1.0,70%
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,v1.0,74%
...,...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,v1.0,60%
9996,NaT,STATION_A,480.22,24.455721,1009.338727,v1.0,76%
9997,NaT,STATION_A,470.45,24.299476,1013.216474,v1.0,91%
9998,NaT,STATION_C,363.91,22.642465,1014.534921,v1.0,54%


# The Cleaned Data Frame

We have now finished cleaning our data frame!

In [56]:
df_clean = df6.copy()
df_clean

,recorded_at,station_code,power_kwh,temp_celsius,baro_pressure_hpa,firmware,battery_pct
0,2024-03-01 00:00:00+00:00,STATION_B,413.61,NaN,1016.023518,v1.2,88%
1,2024-03-01 00:05:00+00:00,STATION_A,535.03,24.777070,1017.724282,v1.0,56%
2,2024-03-01 00:10:00+00:00,UNKNOWN,431.52,28.232964,1011.302091,v1.0,64%
3,2024-03-01 00:15:00+00:00,STATION_C,456.54,22.708422,1007.182413,v1.0,70%
4,2024-03-01 00:20:00+00:00,STATION_B,508.65,26.940694,1010.992362,v1.0,74%
...,...,...,...,...,...,...,...
9995,NaT,UNKNOWN,437.93,20.386788,1014.569947,v1.0,60%
9996,NaT,STATION_A,480.22,24.455721,1009.338727,v1.0,76%
9997,NaT,STATION_A,470.45,24.299476,1013.216474,v1.0,91%
9998,NaT,STATION_C,363.91,22.642465,1014.534921,v1.0,54%
